# 02: A deep demo of every stock global fit

**This notebook targets the development stack (LATW `dev` branch).** It
builds and runs the *installed* stock global fits from the development
LISA Analysis Tools packages (`LISAanalysistools/install.sh`), not the pip
releases. No Colab:

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
os.environ.setdefault("MAKE_DIAGNOSTIC_PLOTS", "0")

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from copy import deepcopy
from lisatools.utils.constants import *

from lisatools.globalfit.stock import erebor

# The two readouts this gallery leans on are methods on the installed classes,
# not helpers defined here:
#   fit.describe()          -- the configuration; once built, each branch also
#                              reports the *Setup it resolved into (+ sub-bands).
#                              describe(full=True) dumps EVERY field.
#   curr.summarize_run(...) -- what the sampler produced: log-like + per-branch
#                              chain shapes and alive-leaf counts, read off the
#                              fit's own HDF backend.

# Each variant writes to its own file under a fresh gallery dir, so the
# several stock fits in this single notebook never collide on one HDF5
# backend (eryn rejects reusing a backend whose branch set differs).
import shutil
GALLERY_DIR = "./gf_output_gallery/"
shutil.rmtree(GALLERY_DIR, ignore_errors=True)

# Slightly larger default text so labels/ticks/titles read clearly in the
# rendered figures.
plt.rcParams.update({"font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12,
                     "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11})

cupy not found, using numpy instead. This will be very slow for large runs. Please install cupy and a compatible CUDA version for GPU acceleration.


Right after the quickstart, this notebook is the **gallery**: an early,
catalogue-style overview that builds every registered stock variant through
its `*_lite` twin, reads its structure, and points out how each one differs
from the six-branch `all_sources` flagship. The shared machinery underneath
these variants — the data layer, the settings pyramid, and the recipe of
moves — is unpacked later in [`08`](08_StockGlobalFitsInDepth.ipynb).

Every run here is deliberately tiny -- **synthetic in-process data, two
sampler iterations, laptop grids** -- so the whole notebook fits its time
budget and needs no external data files. None of these short chains is
converged; they exercise the pipeline and show the shape of each fit's
output. The one-line **production** launch for each variant is given in
its section.

### How to read this notebook

One `##` section per variant. Each builds the lite twin and prints how it
is composed (`describe()` / `list_moves()` / branch list). The three light
variants (`gb_no_fg`, `noise_only`, `noise_sgwb`) also **run** a couple of
iterations and read the branches back with the `summarize` helper. The two
heaviest -- `full_year_combined` and the six-branch `all_sources` -- we
**build and inspect** rather than sample (the full `all_sources` build +
run + structured-output read is notebook 01's job; re-running it here would
duplicate it and strain a laptop).


## `gb_no_fg` — galactic binaries only

**TL;DR.** A GB-only reversible-jump fit with a **fixed PSD** and **no
foreground branch**, restricted to f > 6 mHz so unresolved confusion is
out of band. The single `gb` branch uses the WDM chunked-heterodyne
likelihood. Supported data: `mojito` (default) / `synthetic`.

In [2]:
fit = erebor.gb_no_fg_lite(file_store_dir=GALLERY_DIR, base_file_name="gallery_gb_no_fg_lite")
fit.general.data_mode = "synthetic"    # 2-source in-process GB stream
fit.general.num_iterations = 2          # explicit (the lite preset pins 10)
fit.gb.nleaves_max = 4                   # cap the RJ leaf budget for speed
fit.gb.num_repeat_proposals = 1
print(fit.describe())

GBNoForegroundLiteGlobalFit (gb_no_fg_lite) — Laptop-smoke twin of gb_no_fg: two-week span, 10 iterations, 4 walkers x 2 temps, 2 GB repeat proposals, CPU.
  built: False
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 2.5
    num_iterations = 2
    file_store_dir = './gf_output_gallery/'
    base_file_name = 'gallery_gb_no_fg_lite'
    gpus = None
    gpu_backend = 'auto'
    tobs_target = 1209600.0
    min_freq = 0.006
    max_freq = 0.025
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.05
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['gb']
    gb: GBNoFgGBSettings
  recipe:
    [pe] gb_pe:
        rj_prior  <- stock branch=gb
  setup_function: setup_recipe


In [3]:
curr = fit.build()
fit.run()
curr.summarize_run("gb_no_fg_lite");

2026-07-15 14:31:45,567 - GeneralSetup - DEBUG - Saving h5 backend to ./gf_output_gallery/gallery_gb_no_fg_lite_testing.h5


2026-07-15 14:31:45,568 - GeneralSetup - DEBUG - Saving artifacts to ./gf_output_gallery/gallery_gb_no_fg_lite_artifacts/


2026-07-15 14:31:49,614 - GeneralSetup - INFO - Using fixed PSD kwargs: {'psd_params': [1.5e-11, 3e-15], 'galfor_params': None}


2026-07-15 14:31:49,614 - GeneralSetup - DEBUG - Preprocess setting: plot_folder = ./gf_output_gallery/gallery_gb_no_fg_lite_artifacts/


2026-07-15 14:31:49,615 - GeneralSetup - DEBUG - Preprocess setting: highpass_kwargs = None


...

1it [00:06,  6.11s/it]


100%|██████████| 2/2 [00:15<00:00,  7.47s/it]

100%|██████████| 2/2 [00:15<00:00,  7.75s/it]

2026-07-15 14:32:17,307 - lisatools.globalfit.run - INFO - Residuals saved.


=== gb_no_fg_lite (sampled) ===
branches   : ['gb']
log_like   : (2, 2, 4)  final chain (temp 0): [-6.39 -6.39 -6.39 -6.39]
  gb      chain (2, 2, 4, 4, 8)            alive-leaves(temp 0,last)=0


**How it differs from `all_sources`:** just the one `gb` branch (vs.
six), a *fixed* instrument PSD instead of a fitted `psd`/`galfor`, and a
narrow high-frequency band. The `gb` chain is `(nsteps, ntemps, nwalkers,
nleaves_max, ndim=8)` — a reversible-jump *forest*, so the alive-leaf
count is itself sampled.

**Production launch:** `erebor.gb_no_fg()` (mojito L1 GB galaxy, full band,
`num_iterations` from `NUM_ITERATIONS`); scale with `NWALKERS`,
`TOBS_TARGET`, `GB_MODE=search`.

## `noise_only` — instrument PSD + galactic foreground

**TL;DR.** No GW *source* branches at all: it jointly fits the instrument
**`psd`** (2 params) and the hyperbolic-tangent galactic-confusion
**`galfor`** foreground on a self-consistent synthetic WDM noise
realization, both sampled by the single `PSDMove`. Data: `synthetic` only
(swap `data_processor_class` for real data).

In [4]:
fit = erebor.noise_only_lite(file_store_dir=GALLERY_DIR, base_file_name="gallery_noise_only_lite")
fit.general.num_iterations = 2
print(fit.describe())
print("\nmoves:\n" + fit.list_moves())

NoiseOnlyLiteGlobalFit (noise_only_lite) — Laptop-smoke twin of noise_only: quarter-length time grid, 10 iterations, 4 walkers x 2 temps, CPU.
  built: False
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 5.0
    num_iterations = 2
    file_store_dir = './gf_output_gallery/'
    base_file_name = 'gallery_noise_only_lite'
    gpus = None
    gpu_backend = 'auto'
    tobs_target = 7776000.0
    min_freq = 0.0003
    max_freq = 0.008
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.0
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['psd', 'galfor']
    psd: NoisePSDSettings
    galfor: GalForSettings
  recipe:
    [pe] noise_pe:
        psd_pe  <- stock branch=psd
  setup_function: setup_recipe

moves:
[pe] noise_pe:
    psd_pe  <- stock branch=psd


In [5]:
curr = fit.build()
fit.run()
curr.summarize_run("noise_only_lite");

2026-07-15 14:32:17,814 - lisatools.globalfit.stock.erebor.noise - INFO - noise parameters read from /Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/data/INSTRUMENT/L1/NOISE_731d_2.5s_L1_source0_0_20251206T220508924302Z.h5: Soms_d=1.496182e-11, Sa_a=2.982412e-15


2026-07-15 14:32:17,816 - GeneralSetup - DEBUG - Saving h5 backend to ./gf_output_gallery/gallery_noise_only_lite_testing.h5


2026-07-15 14:32:17,817 - GeneralSetup - DEBUG - Saving artifacts to ./gf_output_gallery/gallery_noise_only_lite_artifacts/


2026-07-15 14:32:17,818 - lisatools.globalfit.preprocessing - INFO - L1DataLoader initialized with data folder: /Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/data


2026-07-15 14:32:17,818 - lisatools.globalfit.preprocessing - INFO - Source types to load: ['NOISE']


...

1it [00:06,  6.10s/it]


100%|██████████| 2/2 [00:11<00:00,  5.97s/it]

100%|██████████| 2/2 [00:11<00:00,  5.94s/it]

2026-07-15 14:33:58,538 - lisatools.globalfit.run - INFO - Residuals saved.


=== noise_only_lite (sampled) ===
branches   : ['psd', 'galfor']
log_like   : (2, 2, 4)  final chain (temp 0): [1961253.54 1893293.63 1832060.29 1944094.72]
  psd     chain (2, 2, 4, 1, 2)            alive-leaves(temp 0,last)=4
  galfor  chain (2, 2, 4, 1, 5)            alive-leaves(temp 0,last)=4


**How it differs from `all_sources`:** the `psd` + `galfor` branches
here are *exactly* the noise branches `all_sources` also carries — but
with no source branches to subtract, so the residual **is** the noise. The
injected truths (`PSD_INJECTION`, `GALFOR_INJECTION`) parameterize the
covariance the data are drawn from and sit inside the sampled priors, so a
converged run recovers them. Both branches are fixed-count (`nleaves_max =
1`), unlike the RJ source forests.

**Production launch:** `erebor.noise_only()` (longer grid, more
iterations).

## `noise_sgwb` — PSD + foreground + a stochastic background

**TL;DR.** `noise_only` plus a third branch: a power-law **`sgwb`**
stochastic gravitational-wave background, fit jointly through the same
`PSDMove`. Data: `synthetic` only.

In [6]:
fit = erebor.noise_sgwb_lite(file_store_dir=GALLERY_DIR, base_file_name="gallery_noise_sgwb_lite")
fit.general.num_iterations = 2
print("branches:", list(fit.branches))
print(fit.list_moves())

branches: ['psd', 'galfor', 'sgwb']
[pe] noise_pe:
    psd_pe  <- stock branch=psd


In [7]:
curr = fit.build()
fit.run()
curr.summarize_run("noise_sgwb_lite");

2026-07-15 14:33:59,471 - lisatools.globalfit.stock.erebor.noise - INFO - noise parameters read from /Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/data/INSTRUMENT/L1/NOISE_731d_2.5s_L1_source0_0_20251206T220508924302Z.h5: Soms_d=1.496182e-11, Sa_a=2.982412e-15


2026-07-15 14:33:59,496 - GeneralSetup - DEBUG - Saving h5 backend to ./gf_output_gallery/gallery_noise_sgwb_lite_testing.h5


2026-07-15 14:33:59,500 - GeneralSetup - DEBUG - Saving artifacts to ./gf_output_gallery/gallery_noise_sgwb_lite_artifacts/


2026-07-15 14:33:59,511 - lisatools.globalfit.preprocessing - INFO - L1DataLoader initialized with data folder: /Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/data


2026-07-15 14:33:59,512 - lisatools.globalfit.preprocessing - INFO - Source types to load: ['NOISE']


...

1it [00:08,  8.57s/it]


100%|██████████| 2/2 [00:16<00:00,  8.32s/it]

100%|██████████| 2/2 [00:16<00:00,  8.25s/it]

2026-07-15 14:35:24,242 - lisatools.globalfit.run - INFO - Residuals saved.


=== noise_sgwb_lite (sampled) ===
branches   : ['psd', 'galfor', 'sgwb']
log_like   : (2, 2, 4)  final chain (temp 0): [1982961.89 1106270.56 1917077.36 1879926.57]
  psd     chain (2, 2, 4, 1, 2)            alive-leaves(temp 0,last)=4
  galfor  chain (2, 2, 4, 1, 5)            alive-leaves(temp 0,last)=4
  sgwb    chain (2, 2, 4, 1, 2)            alive-leaves(temp 0,last)=4


**How it differs from `noise_only`:** one extra `sgwb` branch (a
2-param power law, `log10_A @ 25 Hz` + slope). All three noise branches —
`psd`, `galfor`, `sgwb` — ride the *single* `PSDMove`; there is still no
source branch. The SGWB prior is widened from the GLASS default so it
contains the injected amplitude.

**Production launch:** `erebor.noise_sgwb()`. (The SGWB posterior is not
validated at this smoke size — it is built end-to-end for the machinery.)

## `full_year_combined` — the heavy multi-leaf source branches

**TL;DR.** The MBH + EMRI + SOBBH source branches (no GB, no noise
branches) over a long window, each a **multi-leaf** catalogue-driven
forest. Fixed instrument noise + an annually-modulated galactic foreground
are baked into the sensitivity. Waveform paths: **SOBBH TDI-on-the-fly**,
**MBH + EMRI legacy**. Data: `mojito` (default) / `synthetic`.

In [8]:
fit = erebor.full_year_combined_lite(file_store_dir=GALLERY_DIR, base_file_name="gallery_full_year_combined_lite")
fit.general.data_mode = "synthetic"     # build MBH/EMRI/SOBBH streams in-process
fit.general.num_iterations = 2
print(fit.describe())

FullYearCombinedLiteGlobalFit (full_year_combined_lite) — Laptop-smoke twin of full_year_combined: one month instead of a year, 10 iterations, 4 walkers x 2 temps, CPU.
  built: False
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 2.5
    num_iterations = 2
    file_store_dir = './gf_output_gallery/'
    base_file_name = 'gallery_full_year_combined_lite'
    gpus = None
    gpu_backend = 'auto'
    tobs_target = 2592000.0
    min_freq = 0.0001
    max_freq = 0.025
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.0
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['emri']
    emri: SourceEMRISettings
  recipe:
    [pe] full_pe:
        emri_pe  <- stock branch=emri
  setup_function: setup_recipe


In [9]:
# Built, not sampled here: build() already resolves every branch, loads/builds
# the data, and materializes the recipe -- which is what this gallery is about.
# The two heaviest fits are only inspected; the full build+run+read cycle is in
# notebook 01. Once built, describe() reports each branch's resolved *Setup.
curr = fit.build()          # full_year_combined_lite
print(curr.describe())

2026-07-15 14:35:24,401 - GeneralSetup - DEBUG - Saving h5 backend to ./gf_output_gallery/gallery_full_year_combined_lite_testing.h5


2026-07-15 14:35:24,402 - GeneralSetup - DEBUG - Saving artifacts to ./gf_output_gallery/gallery_full_year_combined_lite_artifacts/


2026-07-15 14:35:24,790 - few - DEBUG - Initializing integrator with func=<class 'few.trajectory.ode.flux.KerrEccEqFlux'>, downsample=None, buffer_length=10000


2026-07-15 14:35:24,803 - few - DEBUG - Configuration file not found in '/Users/mkatz/Research/lisa_sprint_2026/LATW/tutorials/few.ini'


2026-07-15 14:35:24,804 - few - DEBUG - Configuration file not found in '/Users/mkatz/Library/Application Support/few.ini'


...

2026-07-15 14:36:42,829 - GeneralSetup - INFO - Domain setting: WAVELET_FILTER_CONSTANT = 4


2026-07-15 14:36:42,831 - GeneralSetup - INFO - Domain setting: omega = [-1.96349541e-04 -1.90213618e-04 -1.84077695e-04 -1.77941771e-04
 -1.71805848e-04 -1.65669925e-04 -1.59534002e-04 -1.53398079e-04
 -1.47262156e-04 -1.41126232e-04 -1.34990309e-04 -1.28854386e-04
 -1.22718463e-04 -1.16582540e-04 -1.10446617e-04 -1.04310694e-04
 -9.81747704e-05 -9.20388473e-05 -8.59029241e-05 -7.97670010e-05
 -7.36310778e-05 -6.74951547e-05 -6.13592315e-05 -5.52233084e-05
 -4.90873852e-05 -4.29514621e-05 -3.68155389e-05 -3.06796158e-05
 -2.45436926e-05 -1.84077695e-05 -1.22718463e-05 -6.13592315e-06
  0.00000000e+00  6.13592315e-06  1.22718463e-05  1.84077695e-05
  2.45436926e-05  3.06796158e-05  3.68155389e-05  4.29514621e-05
  4.90873852e-05  5.52233084e-05  6.13592315e-05  6.74951547e-05
  7.36310778e-05  7.97670010e-05  8.59029241e-05  9.20388473e-05
  9.81747704e-05  1.04310694e-04  1.10446617e-04  1.16582540e-04
  1.22718463e-04  1.28854386e-04  1.34990309e-04  1.41126232e-04
  1.47262156e-04  

2026-07-15 14:36:42,832 - GeneralSetup - INFO - Domain setting: window = [0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 5.13469607e-02 6.99372589e-01 2.98893403e+00
 7.89320471e+00 1.58759553e+01 2.65857720e+01 3.87487505e+01
 5.04626504e+01 5.99290624e+01 6.62280523e+01 6.95766644e+01
 7.09271140e+01 7.13023453e+01 7.13615377e+01 7.13649462e+01
 7.13649646e+01 7.13649646e+01 7.13649646e+01 7.13649646e+01
 7.13649646e+01 7.13649646e+01 7.13649646e+01 7.13649646e+01
 7.13649646e+01 7.13649646e+01 7.13649646e+01 7.13649646e+01
 7.13649646e+01 7.13649646e+01 7.13649646e+01 7.13649646e+01
 7.13649646e+01 7.13649462e+01 7.13615377e+01 7.13023453e+01
 7.09271140e+01 6.95766644e+01 6.62280523e+01 5.99290624e+01
 5.04626504e+01 3.87487505e+01 2.65857720e+01 1.58759553e+01
 7.89320471e+00 2.98893403e+00 6.99372589e-01 5.13469607e-02
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e

2026-07-15 14:36:42,853 - EMRISetup - INFO - Using betas: [1.         0.83333333 0.69444444 0.5787037  0.48225309 0.40187757
 0.33489798 0.27908165 0.23256804 0.1938067  0.16150558 0.13458799
 0.11215665 0.09346388 0.07788657 0.06490547 0.05408789 0.04507324
 0.03756104 0.03130086 0.02608405 0.02173671 0.01811393 0.01509494] in EMRI branch


FullYearCombinedLiteGlobalFit (full_year_combined_lite) — Laptop-smoke twin of full_year_combined: one month instead of a year, 10 iterations, 4 walkers x 2 temps, CPU.
  built: True
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 2.5
    num_iterations = 2
    file_store_dir = './gf_output_gallery/'
    base_file_name = 'gallery_full_year_combined_lite'
    gpus = None
    gpu_backend = 'auto'
    tobs_target = 2592000.0
    min_freq = 0.0001
    max_freq = 0.025
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.0
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['emri']
    emri: SourceEMRISettings  [built: EMRISetup]
  recipe:
    [pe] full_pe:
        emri_pe  <- stock branch=emri
  setup_function: setup_recipe


**Multi-leaf specifics:** each source branch is a reversible-jump
forest — `nleaves_max > 1` — because the number of MBHs / EMRIs / SOBBHs
is unknown and sampled. The engine registers a `signal_gen` on each branch
so it builds and *subtracts* every leaf's template from the shared
residual under the hood (the `signal_gen` path from
[`01` §Per-branch source info](01_GlobalFitQuickstart.ipynb)); the moves
then resample each leaf. `all_sources` reuses this exact source setup —
`full_year_combined` is just those three branches on their own over a full
year.

**Production launch:** `erebor.full_year_combined()` (mojito L1, a full
year; `TOBS_TARGET`, `CHOP_WINDOW=1` for a merger-centered MBH snippet).

## `all_sources` -- the six-branch flagship

**TL;DR.** The real joint global fit: **all six branches** -- `gb`, `psd`,
`galfor`, `mbh`, `emri`, `sobbh` -- composed from the exact per-branch
setups of the variants above (the `gb_no_fg` GB down to 0.1 mHz, the noise
`psd`/`galfor`, the `full_year` MBH/EMRI/SOBBH). This is the long pole; we
**build and inspect** it here and leave the full run to notebook 01. Data:
`mojito` (default) / `synthetic` / `sangria`.


In [10]:
fit = erebor.all_sources_lite(file_store_dir=GALLERY_DIR, base_file_name="gallery_all_sources_lite")
fit.general.data_mode = "synthetic"     # all six streams in-process, no external data
fit.general.num_iterations = 2
print(fit.describe())

AllSourcesLiteGlobalFit (all_sources_lite) — Laptop-smoke twin of all_sources: the same six branches and machinery on a small WDM grid (nf=720, nt=180), 10 iterations, 4 walkers x 2 temps, narrow GB band, CPU. Scale any knob back up to reach the full model (and vice-versa via all_sources(lite=True)).
  built: False
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 5.0
    num_iterations = 2
    file_store_dir = './gf_output_gallery/'
    base_file_name = 'gallery_all_sources_lite'
    gpus = None
    gpu_backend = 'auto'
    tobs_target = 7776000.0
    min_freq = 0.0001
    max_freq = 0.025
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.0
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['gb', 'psd', 'galfor', 'mbh', 'emri', 'sobbh']
    gb: AllSourcesGBSettings
    psd: NoisePSDSettings
    galfor: GalForSettings
    mbh: SourceMBHSettings
    emri: SourceEMRISettings
    sobbh: SourceSOBBHSettings
  recipe:
    [

In [11]:
# Built, not sampled here: build() already resolves every branch, loads/builds
# the data, and materializes the recipe -- which is what this gallery is about.
# The two heaviest fits are only inspected; the full build+run+read cycle is in
# notebook 01. Once built, describe() reports each branch's resolved *Setup.
curr = fit.build()          # all_sources_lite
print(curr.describe())

2026-07-15 14:36:43,114 - lisatools.globalfit.stock.erebor.fit - INFO - add_instrument_noise=True resolved to 'synthetic'.


2026-07-15 14:36:43,117 - GeneralSetup - DEBUG - Saving h5 backend to ./gf_output_gallery/gallery_all_sources_lite_testing.h5


2026-07-15 14:36:43,117 - GeneralSetup - DEBUG - Saving artifacts to ./gf_output_gallery/gallery_all_sources_lite_artifacts/


No CuPy


2026-07-15 14:36:48,835 - lisatools.sources.waveformbase - WARNING - No Orbits object provided. Using default EqualArmlengthOrbits.


...

2026-07-15 14:38:34,999 - PSDSetup - INFO - Using custom priors for PSD branch


2026-07-15 14:38:35,011 - MBHSetup - DEBUG - Decide how to treat fdot prior


2026-07-15 14:38:35,016 - EMRISetup - INFO - Using betas: [1.         0.83333333 0.69444444 0.5787037  0.48225309 0.40187757
 0.33489798 0.27908165 0.23256804 0.1938067  0.16150558 0.13458799
 0.11215665 0.09346388 0.07788657 0.06490547 0.05408789 0.04507324
 0.03756104 0.03130086 0.02608405 0.02173671 0.01811393 0.01509494] in EMRI branch


2026-07-15 14:38:35,029 - SOBBHSetup - INFO - Using betas: [1.         0.83333333 0.69444444 0.5787037  0.48225309 0.40187757
 0.33489798 0.27908165 0.23256804 0.1938067  0.16150558 0.13458799
 0.11215665 0.09346388 0.07788657 0.06490547 0.05408789 0.04507324
 0.03756104 0.03130086 0.02608405 0.02173671 0.01811393 0.01509494] in SOBBH branch


AllSourcesLiteGlobalFit (all_sources_lite) — Laptop-smoke twin of all_sources: the same six branches and machinery on a small WDM grid (nf=720, nt=180), 10 iterations, 4 walkers x 2 temps, narrow GB band, CPU. Scale any knob back up to reach the full model (and vice-versa via all_sources(lite=True)).
  built: True
  headline:
    nwalkers = 4
    ntemps = 2
    dt = 5.0
    num_iterations = 2
    file_store_dir = './gf_output_gallery/'
    base_file_name = 'gallery_all_sources_lite'
    gpus = None
    gpu_backend = 'auto'
    tobs_target = 7776000.0
    min_freq = 0.0001
    max_freq = 0.025
    tdi_chan = 'XYZ'
    window_tukey_alpha = 0.0
    mojito_data_path = '/Users/mkatz/.mojito_cache/brickmarket/mojito_light_v1_0_0/'
    use_gpu = False
  branches: ['gb', 'psd', 'galfor', 'mbh', 'emri', 'sobbh']
    gb: AllSourcesGBSettings  [built: GBSetup, 64 sub-bands]
    psd: NoisePSDSettings  [built: PSDSetup]
    galfor: GalForSettings  [built: GalForSetup]
    mbh: SourceMBHSettings  [b

**How it composes the others:** `all_sources` is not a new pipeline —
it is the *union* of the branch setups you just saw. Its `gb` branch is
the `gb_no_fg` setup (widened to the full band), its `psd`/`galfor` are the
`noise` setup, and its `mbh`/`emri`/`sobbh` are the `full_year` setup —
all six sampled together against one shared residual by one combined PE
stage (`fit.list_moves()`). That composition is the whole design: each
specialized variant is a slice of `all_sources`, so validating a branch in
isolation validates it in the joint fit.

**Production launch:** `erebor.all_sources()` (mojito L1 default; ~90-day
WDM grid, `num_iterations` from `NUM_ITERATIONS`, GPU via `USE_GPU=1` /
`GPUS`). See `LISAanalysistools/docs/global-fit-launch.md`.

In [12]:
print(fit.list_moves())

[pe] full_pe:
    psd_pe  <- stock branch=psd
    mbh_pe  <- stock branch=mbh
    emri_pe  <- stock branch=emri
    sobbh_pe  <- stock branch=sobbh
    rj_prior  <- stock branch=gb


## Recap

| variant | branches | data modes | notes |
|---|---|---|---|
| `gb_no_fg` | `gb` | mojito / synthetic | fixed PSD, f > 6 mHz, RJ |
| `noise_only` | `psd`, `galfor` | synthetic | noise is the signal |
| `noise_sgwb` | `psd`, `galfor`, `sgwb` | synthetic | + power-law SGWB |
| `full_year_combined` | `mbh`, `emri`, `sobbh` | mojito / synthetic | multi-leaf sources |
| `all_sources` | all six | mojito / synthetic / sangria | the joint fit |

Every full model has a `*_lite` twin (used throughout here) and scales to
production by turning the knobs back up — see
[`01` §Lite ↔ heavy](01_GlobalFitQuickstart.ipynb) and
[`08`](08_StockGlobalFitsInDepth.ipynb) for the machinery. Launch details:
`LISAanalysistools/docs/global-fit-launch.md`.